In [1]:
question = globals().get("question", "")

In [2]:
print("QUESTION (inyectada):", question)


QUESTION (inyectada): 


In [3]:
import os
import requests
import pandas as pd
from tqdm import tqdm
import time
import pg8000
from openai import OpenAI
import re
import warnings
from IPython.display import display, clear_output

try:
    import ipywidgets as widgets
except Exception:
    widgets = None

warnings.filterwarnings(
    "ignore",
    message="pandas only supports SQLAlchemy connectable",
    category=UserWarning,
)


In [4]:
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

conn = pg8000.connect(
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT", "5432")),
    database=os.getenv("DB_NAME", "postgres"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),

)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("Conectado")

Conectado


In [5]:
def construir_prompt(pregunta):

    return f"""
Eres un experto en SQL para PostgreSQL.

Tabla: juego

Columnas reales:
- name
- rating
- metacritic
- ratings_count
- reviews_text_count
- released
- added
- genres
- platforms

Reglas:
- SOLO devuelve SQL
- NO expliques nada
- Solo SELECT
- Solo puedes consultar la tabla juego
- Siempre incluye LIMIT

Búsqueda por nombre (MUY IMPORTANTE):
- Si el usuario pregunta por un juego concreto (ej: "que me puedes decir sobre Super Mario World"), debes buscarlo por la columna name usando ILIKE.
- Usa las palabras importantes del nombre y crea condiciones AND. Ejemplo:
  WHERE name ILIKE '%mario%' AND name ILIKE '%world%'
- Usa LIMIT 5 en este tipo de búsquedas.

Año:
- NO existe la columna year. Si el usuario pregunta por un año (ej. 2000), usa:
  EXTRACT(YEAR FROM released) = 2000
  y filtra released IS NOT NULL.

Interpretaciones:
- "mas jugado/mas usado/mas popular" => mayor added
- "mas antiguo" => released ASC filtrando NULL
- "mas moderno/mas reciente/nuevo" => released DESC filtrando NULL
- Si pregunta "antiguo y moderno" o pide dos extremos, usa UNION ALL con dos SELECT.

Ejemplos:

Pregunta: que me puedes decir sobre Super Mario World
SQL:
SELECT * FROM juego
WHERE name ILIKE '%mario%'
  AND name ILIKE '%world%'
LIMIT 5;

Pregunta: juego del año 2000
SQL:
SELECT * FROM juego
WHERE released IS NOT NULL
  AND EXTRACT(YEAR FROM released) = 2000
LIMIT 10;

Pregunta: juego mas jugado
SQL:
SELECT * FROM juego
ORDER BY added DESC
LIMIT 1;

Pregunta: juego mas antiguo y mas moderno
SQL:
(SELECT * FROM juego
 WHERE released IS NOT NULL
 ORDER BY released ASC
 LIMIT 1)
UNION ALL
(SELECT * FROM juego
 WHERE released IS NOT NULL
 ORDER BY released DESC
 LIMIT 1)
LIMIT 2;

Pregunta:
{pregunta}
"""

In [ ]:
def _extraer_contexto_tiempo(pregunta: str):
    p = (pregunta or "").lower()

    disparadores_todos = [
        "todos los tiempos",
        "de todos los tiempos",
        "de la historia",
        "de toda la historia",
        "todos los años",
        "toda la vida",
    ]
    if any(d in p for d in disparadores_todos):
        return {"modo": "all_time", "year": None}

    m = re.search(r"\b(19[8-9]\d|20\d{2}|2100)\b", p)
    if m:
        return {"modo": "year", "year": int(m.group(0))}

    return {"modo": "none", "year": None}


def construir_prompt(pregunta):

    ctx = _extraer_contexto_tiempo(pregunta)

    if ctx["modo"] == "all_time":
        instruccion_tiempo = (
            "- El usuario pidió 'de todos los tiempos': NO apliques ningún filtro por año.\n"
            "- Aun así, si vas a ordenar por antigüedad/recencia usa released IS NOT NULL."  
        )
        anio_detectado = "TODOS"
    elif ctx["modo"] == "year":
        instruccion_tiempo = (
            "- Se detectó un año explícito en la pregunta: debes filtrar SIEMPRE por ese año.\n"
            "- Usa released IS NOT NULL y EXTRACT(YEAR FROM released) = <AÑO_DETECTADO>."
        )
        anio_detectado = str(ctx["year"])
    else:
        instruccion_tiempo = (
            "- Si el usuario pregunta explícitamente por un año, filtra por ese año con EXTRACT(YEAR FROM released).\n"
            "- Si no pide año, no inventes filtros temporales."
        )
        anio_detectado = "(no)"

    return f"""
Eres un experto en SQL para PostgreSQL.

Tabla: juego

Columnas reales:
- name
- rating
- metacritic
- ratings_count
- reviews_text_count
- released
- added
- genres
- platforms

Reglas:
- SOLO devuelve SQL
- NO expliques nada
- Solo SELECT
- Solo puedes consultar la tabla juego
- Siempre incluye LIMIT

Búsqueda por nombre (MUY IMPORTANTE):
- Si el usuario pregunta por un juego concreto (ej: "que me puedes decir sobre Super Mario World"), debes buscarlo por la columna name usando ILIKE.
- Usa las palabras importantes del nombre y crea condiciones AND. Ejemplo:
  WHERE name ILIKE '%mario%' AND name ILIKE '%world%'
- Usa LIMIT 5 en este tipo de búsquedas.

Año / tiempo:
{instruccion_tiempo}

Contexto detectado:
- AÑO_DETECTADO = {anio_detectado}

Nota técnica:
- NO existe la columna year. Si el usuario pregunta por un año (ej. 2000), usa:
  EXTRACT(YEAR FROM released) = 2000
  y filtra released IS NOT NULL.

Interpretaciones:
- "mas jugado/mas usado/mas popular" => mayor added
- "mas antiguo" => released ASC filtrando NULL
- "mas moderno/mas reciente/nuevo" => released DESC filtrando NULL
- Si pregunta "antiguo y moderno" o pide dos extremos, usa UNION ALL con dos SELECT.

Ejemplos:

Pregunta: que me puedes decir sobre Super Mario World
SQL:
SELECT * FROM juego
WHERE name ILIKE '%mario%'
  AND name ILIKE '%world%'
LIMIT 5;

Pregunta: juego del año 2000
SQL:
SELECT * FROM juego
WHERE released IS NOT NULL
  AND EXTRACT(YEAR FROM released) = 2000
LIMIT 10;

Pregunta: juego mas jugado
SQL:
SELECT * FROM juego
ORDER BY added DESC
LIMIT 1;

Pregunta: juego mas antiguo y mas moderno
SQL:
(SELECT * FROM juego
 WHERE released IS NOT NULL
 ORDER BY released ASC
 LIMIT 1)
UNION ALL
(SELECT * FROM juego
 WHERE released IS NOT NULL
 ORDER BY released DESC
 LIMIT 1)
LIMIT 2;

Pregunta:
{pregunta}
"""

In [ ]:
def ejecutar_pregunta(pregunta, output=None):
    def _run():
        p = (pregunta or "").strip()

        if not p:
            print("Escribe una pregunta.")
            return

        if not validar_pregunta(p):
            print("Solo puedo responder preguntas relacionadas con videojuego")
            return

        sql = generar_sql(p)
        sql = limpiar_sql(sql)

        valido, mensaje = validar_sql(sql)
        if not valido:
            print(mensaje)
            print(f"SQL generado: {sql}")
            return

        try:
            df = pd.read_sql(sql, conn)
        except Exception as e:
            print("Error ejecutando el SQL en la base de datos")
            print(f"SQL: {sql}")
            print(f"Error: {e}")
            return

        if df.empty:
            keywords = extraer_keywords_busqueda(p)
            queries_fb = construir_fallbacks_nombre(keywords, limit=5)

            encontrado = False
            for sql_fb in queries_fb:
                try:
                    df_fb = pd.read_sql(sql_fb, conn)
                except Exception:
                    continue

                if not df_fb.empty:
                    sql = sql_fb
                    df = df_fb
                    encontrado = True
                    break

            if not encontrado:
                print("No encontré resultados para esa pregunta")
                return

        respuesta = generar_respuesta(p, sql, df)
        print(respuesta)

    if output is None:
        _run()
    else:
        with output:
            clear_output()
            _run()


pregunta = (question or "").strip()
if pregunta:
    ejecutar_pregunta(pregunta)


In [ ]:
if widgets is not None:
    pregunta_input = widgets.Text(
        placeholder="Ej: ¿Qué me puedes decir sobre Super Mario World?",
        description="Pregunta:",
        layout=widgets.Layout(width="80%"),
    )
    boton = widgets.Button(description="Enviar")
    output = widgets.Output()

    def on_click(b):
        ejecutar_pregunta(pregunta_input.value, output=output)

    boton.on_click(on_click)

    display(pregunta_input, boton, output)
else:
    print("ipywidgets no está disponible en este kernel. Ejecuta con una variable 'question' inyectada o instala ipywidgets.")

Text(value='', description='Pregunta:', layout=Layout(width='80%'), placeholder='Ej: ¿Qué me puedes decir sobr…

Button(description='Enviar', style=ButtonStyle())

Output()